# Suite PAR — Parametrics

A parameter is a variable a schema declares (`OfParameter`), determined where the schema is referred to: `OfApply`
applies a parametric schema to arguments, by name or in order, partially, or with arguments that depend on the
referring schema's own parameters. Where a width or an extent's bound stands, a term may stand instead: a dialect's
term, or its neutral `Form`. Data of an application is data of its applied schema, and modules write parameters,
terms and applications and read them back.

In [ ]:
from dataclasses import dataclass

from mbse.Schemas.Framework import Bindings as Bd, Comparison as C, JSON, Modules, Plain, Proxies, Schemas as S, Validators, YAML
from support import raises, text

Form = S.Form.Data
int_ = lambda t: t.as_native(int)  # noqa: E731


def var(name):
    """A variable's neutral form."""
    return Form("variable", {"name": name})


class Basic:
    """A stand-in for a dialect of mbse-expressions: its terms give their forms and their dialect."""

    @staticmethod
    def name():
        return "basic"


class Var:
    def __init__(self, name):
        self.name = name

    def form(self):
        return Form("variable", {"name": self.name})

    def dialect(self):
        return Basic


class Times:
    def __init__(self, a, b):
        self.a, self.b = a, b

    def form(self):
        return Form("operation", {"name": "mul"}, (self.a, self.b))

    def dialect(self):
        return Basic


Matrix = S.OfObject.Builder().name("Matrix").parameters(lambda p: p.name("rows").of(int_), lambda p: p.name("cols").of(int_)).properties(
    lambda p: p.name("cells").of(lambda t: t.as_indexed(
        lambda i: i.of(lambda x: x.as_native(float)).extent(1, Times(Var("rows"), Var("cols")))))).create()
Word = S.OfNative.Builder().name("Word").type(int).parameters(lambda p: p.name("w").of(int_)).bits(var("w")).create()

## PAR-01 · Every kind of schema declares parameters, in order: each a name, a type (or none) and a description

In [ ]:
n = lambda p: p.name("n").of(int_)  # noqa: E731
declared = [S.OfNative.Builder().type(int).parameters(n).create(), S.OfObject.Builder().parameters(n).create(),
            S.OfRelation.Builder().links("a", "b").parameters(n).create(),
            S.OfUnion.Builder().parameters(n).branches(text("a"), text("b")).create(),
            S.OfIntersection.Builder().parameters(n).parts(text("a"), text("b")).create(),
            S.OfIndexed.Builder().parameters(n).of(int_).create(),
            S.OfApply.Builder().parameters(n).of(S.OfObject.Builder().create()).create()]
assert all(list(s.parameters) == ["n"] and s.validate() == [] for s in declared)
assert S.OfObject.Builder().create().parameters == {} and S.OfNative.Data(int).parameters == {}  # none unless declared
assert list(Matrix.parameters) == ["rows", "cols"] and isinstance(Matrix.parameters["cols"], S.OfParameter.Data)
assert Matrix.parameters["rows"].type == S.OfNative.Data(int) and Matrix.validate() == []
described = S.OfObject.Builder().parameters(lambda p: p.name("n").of(int_).description("How many")).create()
assert described.parameters["n"].description == "How many"
untyped = S.OfObject.Builder().parameters(lambda p: p.name("x")).create()
assert untyped.parameters["x"].type is None and untyped.validate() == []  # a parameter of any type
assert S.OfObject.Builder().parameters(S.OfParameter.Data("n", S.OfNative.Data(int))).create().parameters["n"].name == "n"
assert (S.OfParameter.Data().name, S.OfParameter.Data().type, S.OfParameter.Data().description) == ("", None, None)
bad = S.OfObject.Builder().parameters(lambda p: p.name("$n").of(int_), S.OfParameter.Data("m", "junk"),
                                      lambda p: p.name("d").of(int_).description(True)).create()
assert bad.validate() == ["name '$n' is reserved: names starting with '$' belong to the wire format",
                          "parameter 'm': not a schema: 'junk'", "parameter 'd': a description is text, got bool"]
# A native's parameters are part of it, as its name is; a parameter compares by what it holds
width = lambda: S.OfNative.Builder().type(int).parameters(lambda p: p.name("w").of(int_)).create()  # noqa: E731
assert width() == width() and width() != S.OfNative.Data(int)
any_width = lambda: S.OfNative.Builder().type(int).parameters(lambda p: p.name("w")).create()  # noqa: E731
assert any_width() == any_width() and any_width() != width()
assert S.OfNative.Builder(width()).clone() == width()

## PAR-02 · Terms stand where a width or an extent's bound does: a dialect's term, or its neutral form

In [ ]:
rows = var("rows")
assert rows == Form("variable", {"name": "rows"}) and rows.arguments == () and rows.dialect is None
product = S.Form.of(Times(Var("rows"), Var("cols")))
assert product == Form("operation", {"name": "mul"}, (rows, var("cols")), "basic")  # its dialect at its root only
assert S.Form.of(Times(var("rows"), Var("cols"))) == product  # a form within a term stays as it is
assert Form("b", {"x": b"\x01"}) == Form("b", {"x": b"\x01"}) != Form("b", {"x": b"\x02"})
assert S.Form.of(product) is product and S.Form.is_term(Var("n")) and S.Form.is_term(rows) and not S.Form.is_term(3)
with raises(TypeError, match="not a term: 3"):
    S.Form.of(3)
assert Word.bits == var("w") and Word.validate() == []
assert S.OfNative.Builder().type(int).bytes(Var("w")).create().validate() == []  # a dialect's term is its dialect's to check
cells = Matrix.properties["cells"].type
assert S.Form.of(cells.extent.maximum) == product and cells.validate() == []
assert cells.minimum == 1 and cells.capacity is None  # a capacity needs int bounds
assert S.OfIndexed.Builder().of(int_).extent(rows, 9).create().minimum == 0  # where the minimum is a term
assert S.OfNative.Builder().type(int).bits(Form("", {"x": None}, (None,), True)).create().validate() == [
    "a width in bits: a term needs a kind, got ''", "a width in bits: a term's dialect is a name, got True",
    "a width in bits: attribute 'x' is not a native value: None", "a width in bits: argument 0 is not a term: None"]
assert S.OfNative.Builder().type(int).bits(Form("f", {}, (Form(""),))).create().validate() == [
    "a width in bits: argument 0: a term needs a kind, got ''"]
assert S.OfNative.Builder().type(int).bits("w").create().validate() == ["a width in bits must be a positive int or a term, got 'w'"]
assert S.OfIndexed.Builder().of(int_).extent("a", 3).create().validate() == [
    "an extent's minimum and maximum are ints or terms, got 'a' and 3"]
assert S.OfIndexed.Builder().of(int_).extent(Form(""), 3).create().validate() == ["an extent's minimum: a term needs a kind, got ''"]
assert S.OfIndexed.Builder().of(int_).extent(3, Form("")).create().validate() == ["an extent's maximum: a term needs a kind, got ''"]
assert S.OfIndexed.Builder().of(int_).extent(3, rows).create().validate() == []  # bounds compare only when both are ints

## PAR-03 · `OfApply` applies a parametric schema: arguments by name or in order, partial or dependent

In [ ]:
by_name = S.OfApply.Builder().of(Matrix).argument("cols", 3).argument("rows", 2).create()
in_order = S.OfApply.Builder().of(Matrix).arguments(2, 3).create()
assert by_name.arguments == {"cols": 3, "rows": 2} and in_order.arguments == {"rows": 2, "cols": 3}
assert by_name.of is Matrix and S.structure(by_name) is Matrix and by_name.validate() == in_order.validate() == []
row = S.OfApply.Builder().of(Matrix).arguments(1).create()
assert row.arguments == {"rows": 1} and row.validate() == []  # partial: cols stays unbound
Square = S.OfApply.Builder().name("Square").parameters(lambda p: p.name("n").of(int_)).of(Matrix).arguments(
    var("n"), Var("n")).create()
assert Square.validate() == [] and S.structure(S.OfApply.Builder().of(Square).arguments(4).create()) is Matrix  # dependent
assert S.OfAny.resolve(lambda t: t.as_apply(lambda a: a.of(Matrix))).of is Matrix  # a type, wherever one is expected
assert S.OfApply.resolve(by_name) is by_name and S.structure(Matrix) is Matrix
assert by_name == by_name and by_name != in_order  # an application is equal to itself alone, as other schemas are
with raises(ValueError, match="positional arguments are taken in the applied schema's parameter order; call .of() first"):
    S.OfApply.Builder().arguments(1)
with raises(ValueError, match="the applied schema has 2 parameters, got 3 arguments"):
    S.OfApply.Builder().of(Matrix).arguments(1, 2, 3)
assert S.OfApply.Builder().of(Matrix).argument("depth", 1).argument("rows", "2").argument("cols", None).create().validate() == [
    "the applied schema has no parameter 'depth'", "argument 'rows': expected int, got str",
    "argument 'cols': an argument is a native value or a term, got None"]
Any = S.OfObject.Builder().parameters(lambda p: p.name("x")).properties(lambda p: p.name("$p").of(int_)).create()
assert S.OfApply.Builder().of(Any).argument("x", "text").create().validate() == [
    "of: name '$p' is reserved: names starting with '$' belong to the wire format"]  # any type takes any native value
assert S.OfApply.Builder().of(Any).argument("x", Form("")).create().validate()[0] == "argument 'x': a term: a term needs a kind, got ''"
assert S.OfApply.Data().validate() == ["an application applies a schema, got None"]
loop = S.OfApply.Data()
loop.of = loop
assert loop.validate() == ["an application cannot apply itself"] and S.structure(loop) is None
# Applications of one kind's schemas are of that kind: a union's branches may mix them with the schemas they apply
mixed = S.OfUnion.Builder().branches(lambda b: b.name("a").of(row), lambda b: b.name("b").of(Matrix)).create()
assert mixed.validate() == []
held = S.OfObject.Builder().properties(lambda p: p.name("r").of(S.OfApply.Builder().of(S.OfObject.Builder().ref().create()).create())).create()
assert held.validate() == ["property 'r': a reference object schema cannot be a property's type"]

## PAR-04 · Data of an application is data of its applied schema: proxies, snapshots, validation, comparison, bindings

In [ ]:
Vec = S.OfObject.Builder().name("Vec").parameters(lambda p: p.name("n").of(int_)).properties(
    lambda p: p.name("xs").of(lambda t: t.as_indexed(lambda i: i.of(lambda x: x.as_native(float)).extent(0, var("n"))))).create()
Vec3 = S.OfApply.Builder().of(Vec).arguments(3).create()
Byte = S.OfApply.Builder().of(Word).arguments(8).create()
Real = S.OfApply.Builder().of(S.OfNative.Data(float)).create()
Pair = S.OfRelation.Builder().name("Pair").links("a", "b").properties(lambda p: p.name("weight").of(Byte)).create()
Shape = S.OfObject.Builder().name("Shape").ref().properties(
    lambda p: p.name("v").of(Vec3), lambda p: p.name("w").of(Byte),
    lambda p: p.name("vs").of(lambda t: t.as_indexed(lambda i: i.of(Vec3))),
    lambda p: p.name("by").of(lambda t: t.as_indexed(lambda i: i.key(Real).of(Byte))),
    lambda p: p.name("u").of(S.OfUnion.Builder().branches(lambda b: b.name("a").of(Vec3), lambda b: b.name("b").of(Vec)).create())
    ).relations(lambda a: a.name("pairs").of(Pair).me("a"), lambda a: a.name("paired").of(Pair).me("b")).create()
assert Shape.validate() == [] and Pair.validate() == []
store = Proxies.OfStore()
store.register(Shape)
store.register(Pair)
other = store.Shape().create()
shape = store.Shape().v(lambda v: v.xs([1.0, 2.0])).w(5).vs([lambda v: v.xs([3.0])]).by({1.5: 7}).u(
    lambda u: u.a(lambda v: v.xs([]))).pairs(lambda e: e.b(other).weight(9)).create()
assert (shape.v.xs, shape.w, shape.vs[0].xs, dict(shape.by), shape.u.a.xs) == ((1.0, 2.0), 5, (3.0,), {1.5: 7}, ())
text_ = JSON.ToJSON(store).Reachable(Shape, shape)
assert JSON.ToJSON(store).Reachable(Shape, JSON.FromJSON(store).Reachable(Shape, text_)) == text_
assert '"by": {"1.5": 7}' in text_  # a float key's list is written as a mapping, through its application
assert YAML.ToYAML(store).Reachable(Shape, YAML.FromYAML(store).Reachable(Shape, YAML.ToYAML(store).Reachable(Shape, shape))) == \
    YAML.ToYAML(store).Reachable(Shape, shape)
assert Validators.Validate(store).Reachable(Shape, shape) == []
assert Validators.Validate(store)(Shape, store.Shape().w("five").create()) == ["Shape#0.w: expected int, got str"]
assert C.OfObject(Shape, shape).compare(C.OfObject(Shape, store.Shape(shape).clone())) == 0
assert C.OfObject(Shape, shape).compare(C.OfObject(Shape, store.Shape(shape).w(6).clone())) is None  # not ordered
assert S.OfIndexed.Builder().key(S.OfApply.Builder().of(S.OfNative.Data(int)).create()).of(int_).create().positional


@dataclass(eq=False)
class Box:
    """A class of the program's own, bound to a schema whose properties' types are applications."""

    w: int | None = None
    v: dict | None = None

    def identity(self):
        return id(self)

    def schema_name(self):
        return "Box"

    def owner(self):
        return None

    def accept(self, visitor):
        Bd.accept(BOXED, self, visitor)


BoxSchema = S.OfObject.Builder().name("Box").ref().properties(lambda p: p.name("w").of(Byte), lambda p: p.name("v").of(Vec3)).create()
BOXED = Bd.Binding(BoxSchema, lambda box: Bd.State({"w": box.w, "v": box.v}, {}),
                   lambda state: Box(state.values.get("w"), state.values.get("v")))
bound = Bd.OfStore([(BoxSchema, lambda instance=None: Bd.Builder(BOXED, instance))], [])
box = JSON.FromJSON(bound)(BoxSchema, JSON.ToJSON(bound)(BoxSchema, Box(8, {"xs": [1.0]})))
assert (box.w, box.v) == (8, {"xs": [1.0]}) and Validators.Validate(bound)(BoxSchema, box) == []

## PAR-05 · Modules write parameters, terms and applications, and read them back as forms, or as a dialect's terms

In [ ]:
INT = {"native": {"format": "basic", "name": "int"}}


def form(kind, *arguments, dialect=None, **attributes):
    """A form's plain form: its attributes as `{"str": ...}` values, as all of them are here."""
    return {**({"dialect": dialect} if dialect else {}), "kind": kind,
            **({"attributes": [{"name": k, "value": {"str": v}} for k, v in attributes.items()]} if attributes else {}),
            **({"arguments": list(arguments)} if arguments else {})}


Square = S.OfApply.Builder().name("Square").parameters(lambda p: p.name("n").of(int_)).of(Matrix).arguments(
    var("n"), var("n")).create()
Natives = S.OfObject.Builder().name("Natives").parameters(*[lambda p, x=x: p.name(x) for x in "abcde"]).create()
Holder = S.OfObject.Builder().name("Holder").properties(
    lambda p: p.name("grid").of(lambda t: t.as_apply(lambda a: a.of(Matrix).arguments(2, 3))),
    lambda p: p.name("all").of(S.OfApply.Builder().of(Natives).arguments(1, 1.5, "s", True, b"\x01").create())).create()
Sized = S.OfNative.Builder().name("Sized").type(bytes).bytes(Form("size")).create()
module = Modules.module(store, [Word, Matrix, Square, Natives, Holder, Sized])
plain = Plain.ToPlain(store)(S.Module.Schema, module)
entries = plain["objects"][plain["root"]]["schemas"]
assert entries[0] == {"name": "Word", "schema": {"native": {
    "parameters": [{"name": "w", "type": INT}], "format": "basic", "name": "int", "terms": {"bits": form("variable", name="w")}}}}
assert entries[1]["schema"]["object"]["parameters"] == [{"name": "rows", "type": INT}, {"name": "cols", "type": INT}]
assert entries[1]["schema"]["object"]["properties"][0]["type"]["indexed"]["extent"] == {"minimum": 1, "terms": {"maximum": form(
    "operation", form("variable", name="rows"), form("variable", name="cols"), dialect="basic", name="mul")}}
assert entries[2] == {"name": "Square", "schema": {"apply": {
    "parameters": [{"name": "n", "type": INT}], "of": {"named": {"name": "Matrix"}},
    "arguments": [{"name": "rows", "term": form("variable", name="n")}, {"name": "cols", "term": form("variable", name="n")}]}}}
assert entries[3]["schema"]["object"]["parameters"] == [{"name": x} for x in "abcde"]  # untyped: no type written
assert entries[4]["schema"]["object"]["properties"] == [
    {"name": "grid", "type": {"apply": {"of": {"named": {"name": "Matrix"}}, "arguments": [
        {"name": "rows", "value": {"int": 2}}, {"name": "cols", "value": {"int": 3}}]}}},
    {"name": "all", "type": {"apply": {"of": {"named": {"name": "Natives"}}, "arguments": [
        {"name": "a", "value": {"int": 1}}, {"name": "b", "value": {"float": 1.5}}, {"name": "c", "value": {"str": "s"}},
        {"name": "d", "value": {"bool": True}}, {"name": "e", "value": {"bytes": "AQ=="}}]}}}]
text_ = JSON.ToJSON(store)(S.Module.Schema, module)
back = Modules.schemas(store, JSON.FromJSON(store)(S.Module.Schema, text_))
assert JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, back.values())) == text_
assert back["Word"] == Word and back["Matrix"].properties["cells"].type.extent.maximum == S.Form.of(
    Matrix.properties["cells"].type.extent.maximum)  # read back as forms
assert back["Square"].of is back["Matrix"] and back["Square"].arguments == {"rows": var("n"), "cols": var("n")}
assert back["Holder"].properties["all"].type.arguments == {"a": 1, "b": 1.5, "c": "s", "d": True, "e": b"\x01"}
assert entries[5]["schema"]["native"]["terms"] == {"bytes": {"kind": "size"}} and back["Sized"].bytes == Form("size")
assert back["Natives"].parameters["a"].type is None and all(schema.validate() == [] for schema in back.values())
made = Modules.schemas(store, module, make=lambda f: ("made", f.kind))
assert made["Matrix"].properties["cells"].type.extent.maximum == ("made", "operation") and made["Word"].bits == ("made", "variable")
store.register(Word)  # a named schema resolves in the store
assert Modules.resolve(store, Modules.reference(S.OfApply.Builder().of(Word).argument("w", var("w")).create()),
                       make=lambda f: f.kind).arguments == {"w": "variable"}
with raises(TypeError, match="attribute 'x' of a term is not a native value: None"):
    Modules.reference(S.OfNative.Builder().type(int).bits(Form("v", {"x": None})).create())
with raises(TypeError, match="argument 'x' is not a native value: None"):
    Modules.reference(S.OfApply.Builder().of(Natives).argument("x", None).create())